In [1]:

import pandas as pd

# Sample email dataset
data = {
    "Email_ID": [1, 2, 3, 4, 5, 6],
    "From": [
        "admin@company.com",
        "support@bank.com",
        "security@company.com",
        "alerts@shopping.com",
        "ceo@company.com",
        "info@college.edu"
    ],
    "Return_Path": [
        "admin@company.com",
        "attacker@fake.com",
        "security@company.com",
        "alerts@shopping.com",
        "unknown@fake.com",
        "info@college.edu"
    ],
    "SPF": ["pass", "fail", "pass", "pass", "fail", "pass"],
    "DKIM": ["pass", "fail", "pass", "pass", "pass", "pass"],
    "DMARC": ["pass", "fail", "pass", "pass", "fail", "pass"]
}

df = pd.DataFrame(data)

# Function to detect potential email spoofing
def detect_spoofing(row):
    sender = row["From"].lower().strip()
    return_path = row["Return_Path"].lower().strip()

    # Check sender mismatch and authentication failures
    mismatch = sender != return_path
    failures = sum([
        row["SPF"].lower() == "fail",
        row["DKIM"].lower() == "fail",
        row["DMARC"].lower() == "fail"
    ])

    if row["DMARC"].lower() == "fail" or failures >= 2 or (
        mismatch and row["SPF"].lower() == "fail"
    ):
        return "Potential Spoofing"
    else:
        return "Legitimate"

# Analyze the dataset
df["Result"] = df.apply(detect_spoofing, axis=1)

# Display results
print("EMAIL SPOOFING DETECTION RESULTS")
print(df.to_string(index=False))

# Count suspicious emails
print("\nSummary:")
print(df["Result"].value_counts())


EMAIL SPOOFING DETECTION RESULTS
 Email_ID                 From          Return_Path  SPF DKIM DMARC             Result
        1    admin@company.com    admin@company.com pass pass  pass         Legitimate
        2     support@bank.com    attacker@fake.com fail fail  fail Potential Spoofing
        3 security@company.com security@company.com pass pass  pass         Legitimate
        4  alerts@shopping.com  alerts@shopping.com pass pass  pass         Legitimate
        5      ceo@company.com     unknown@fake.com fail pass  fail Potential Spoofing
        6     info@college.edu     info@college.edu pass pass  pass         Legitimate

Summary:
Result
Legitimate            4
Potential Spoofing    2
Name: count, dtype: int64
